In [1]:
import torch
import torch.nn.functional as F
import plotly.express as px

In [2]:
#words
words= open("names.txt").read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
#mapping
chars= sorted(list(set(''.join(words))))
stoi= {s: i+1 for i, s in enumerate(chars)}
stoi['.']=0
itos= {i:s for s, i in stoi.items()}
stoi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [5]:
vocab_size = 27

In [36]:
#build the dataset

block_size=3

def build_dataset(words):
    X, Y=[], []
    for w in words:

        context=[0]* block_size
        for ch in w + '.':
            ix= stoi[ch]
            X.append(context)
            Y.append(ix)
            context= context[1:]+ [ix]
    X= torch.tensor(X)
    Y= torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1= int(0.8*len(words))
n2= int(0.9*len(words))

Xtr, Ytr= build_dataset(words[:n1])
Xdev, Ydev= build_dataset(words[n1:n2])
Xte, Yte= build_dataset(words[n2:])

torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [37]:
Xtr.shape, Ytr.shape

(torch.Size([182580, 3]), torch.Size([182580]))

In [38]:
n_embd = 10
n_hidden = 200

g=torch.Generator().manual_seed(2147483647)
C=torch.randn((vocab_size,n_embd), generator=g)
W1=torch.randn((n_embd* block_size, n_hidden), generator=g) * (5/3)/ ((n_embd * block_size) ** 0.5)
b1=torch.randn(n_hidden, generator=g) * 0.01
W2=torch.randn((n_hidden, vocab_size)) * 0.01
b2=torch.randn(vocab_size) * 0
parameters=[C, W1, b1, W2, b2]

In [39]:
sum(p.nelement() for p in parameters)

11897

In [40]:
for p in parameters:
    p.requires_grad=True

In [41]:
lre=torch.linspace(-3, 0, 1000)
lrs= 10**lre

In [42]:
lri= []
lossi= []
stepi= []

In [43]:

for i in range(200000):

    #minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (48,))

    #backward pass
    emb= C[Xtr[ix]]
    h= torch.tanh(emb.view(-1,30) @ W1 + b1)
    logits=h @ W2+ b2
    loss= F.cross_entropy(logits, Ytr[ix])
    #print(loss.item())
    
    #forward pass
    for p in parameters:
        p.grad=None 
    loss.backward()

    #update
    #lr= lrs[i]
    
    lr= 0.1 if i<100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # track stats
    #lri.append(lre[i])
    stepi.append(i)
    lossi.append(loss.log10().item())

In [44]:
print(loss.item())

1.9207149744033813


In [45]:
#training loss
emb= C[Xtr]
h= torch.tanh(emb.view(-1,30) @ W1 + b1)
logits=h @ W2+ b2
loss= F.cross_entropy(logits, Ytr)
loss

tensor(2.0309, grad_fn=<NllLossBackward0>)

In [46]:
#validation loss
emb= C[Xdev]
h= torch.tanh(emb.view(-1,30) @ W1 + b1)
logits=h @ W2+ b2
loss= F.cross_entropy(logits, Ydev)
loss

tensor(2.1033, grad_fn=<NllLossBackward0>)

In [47]:
#test loss
emb= C[Xte]
h= torch.tanh(emb.view(-1,30) @ W1 + b1)
logits=h @ W2+ b2
loss= F.cross_entropy(logits, Yte)
loss

tensor(2.1062, grad_fn=<NllLossBackward0>)

In [56]:
px.line(x=stepi, y=lossi)

In [57]:
lossi1 = torch.tensor(lossi).view(-1,200).mean(axis=1)
len1 = len(lossi1)
stepi1 = stepi[:len1]

In [60]:
#Clearer plot
px.line(x=stepi1, y=lossi1)

In [61]:
#sampling from the model

g=torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0]* block_size
    while True:
        emb= C[torch.tensor([context])]
        h = torch.tanh(emb.view(1,-1) @ W1 + b1)
        logits= h @ W2 + b2
        probs= F.softmax(logits, dim=1)
        ix= torch.multinomial(probs, num_samples=1, generator=g).item()
        context= context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    print(''.join(itos[i] for i in out))

carlaizabelle.
khyrmeli.
taty.
skanden.
jazonel.
delsynciereni.
nellara.
chaiir.
kaleigh.
ham.
pois.
quinton.
lilah.
jadiquinterri.
jaristin.
jenni.
sabee.
deci.
abette.
hastraviaan.
